# Lab 4 — Dependency Injection for AI Session Management

Difficulty: Beginner | ~30-35 min | Requires Lab 1

### Step 1: Install Dependencies

We install the exact pinned versions of every library this lab needs. Run this cell first so everything is available for the rest of the notebook.

In [ ]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0

### Step 2: Imports and API Key Setup

Load the necessary modules and read the OpenRouter API key from the `.env` file. We use `load_dotenv()` to load environment variables, then `os.getenv()` to read the key. If the key is not found, we prompt for manual input as a fallback.

In [ ]:
from fastapi import FastAPI, Depends
from typing import Annotated
from fastapi.testclient import TestClient
from dotenv import load_dotenv
from openai import AsyncOpenAI
import os

# Load environment variables from .env file
load_dotenv()

# Read the OpenRouter API key from environment
api_key = os.getenv("OPEN_ROUTER_KEY")

# Fallback: prompt for key if not found in environment
if not api_key:
    api_key = input("Open Router API key: ")

# Module-level dictionary to store conversation histories per session
conversation_store = {}

# Create the FastAPI application instance
app = FastAPI()

### Step 3: Shared Client Dependency

This creates a single `AsyncOpenAI` instance at module level — built once when the module loads, not recreated per request. The `get_client()` function returns this same instance every time FastAPI calls it. This is important because the client holds connection pools and authentication setup that shouldn't be recreated on every single request.

In [ ]:
# Module-level client: built once, shared across all requests
client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

async def get_client():
    """Dependency that returns the shared AsyncOpenAI client."""
    return client

### Step 4: Session ID Dependency

A simple pass-through dependency. FastAPI extracts the `session_id` query parameter and passes it through. This dependency exists so that `get_session` can depend on it — demonstrating dependency chaining.

In [ ]:
async def get_session_id(session_id: str):
    """Pass-through dependency: extracts session_id from query params."""
    return session_id

### Step 5: Session Dependency (yield-dependency)

This is a **yield-dependency**. FastAPI runs the code before `yield` to set up the dependency, yields the value to the endpoint, and then runs the code after `yield` (in the `finally` block) after the endpoint finishes — whether the request succeeded or raised an exception. The `try: yield / finally:` structure guarantees the cleanup code always runs.

In [ ]:
async def get_session(
    session_id: Annotated[str, Depends(get_session_id)]
):
    """Yield-dependency: provides conversation history for a session.
    
    Depends on get_session_id — FastAPI resolves that first, then
    passes the result into this function automatically.
    
    The try/yield/finally structure guarantees the finally block
    runs after the endpoint completes, whether it succeeded or failed.
    """
    # Look up or create the session's history list
    if session_id not in conversation_store:
        conversation_store[session_id] = []
    
    history = conversation_store[session_id]

    try:
        # Yield the history list to the endpoint
        yield history
    finally:
        # This runs AFTER the endpoint finishes — success or failure
        print(f"[Session {session_id}] request finished.")

### Step 6: Chat Endpoint

The `history` and `client` parameters are resolved by FastAPI **before** this function runs. `history` comes from `get_session` (which itself depends on `get_session_id`), and `client` comes from `get_client`. The user message is appended to history **before** the try block so it's recorded even if the LLM call fails.

In [ ]:
@app.post("/chat")
async def chat(
    message: str,
    history: Annotated[list, Depends(get_session)],
    client: Annotated[AsyncOpenAI, Depends(get_client)],
    simulate_error: bool = False
):
    # Append user message BEFORE calling LLM — recorded even on failure
    history.append({
        "role": "user",
        "content": message
    })

    # Prove the client is shared: print its id across multiple calls
    print(f"Client ID: {id(client)}")

    # If simulate_error is True, raise before calling the LLM
    if simulate_error:
        raise ValueError("Simulated Upstream")
    
    # Call the LLM via the injected client
    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=history
    )

    # Extract the answer and append assistant reply to history
    answer = response.choices[0].message.content
    history.append({
        "role": "assistant",
        "content": answer
    })

    return {
        "answer": answer,
        "history": history
    }

### Step 7: TestClient Instantiation

We create a `TestClient` to simulate HTTP requests against our FastAPI app directly in the notebook, without running a live server. One quirk worth knowing: because `/chat` awaits an external async client, running `TestClient` repeatedly in a single notebook session can occasionally raise an "Event loop is closed" error. There is no clean one-line fix worth adding here — if it appears, simply re-run that cell; a fresh event loop is created on retry.

In [ ]:
test_client = TestClient(app)

### Demonstration 1: First Message (New Session)

Send a message to a new session `abs123`. This proves a new session starts with empty history and gets a real answer from the LLM.

In [ ]:
message = "Hello. Where is Paris?"
ID = "abs123"
res = test_client.post(
    "/chat",
    params={
        "message": message,
        "session_id": ID
    })

print(f"\nAnswer: {res.json()['answer']}")

print("\nConversation history:")
for turn in conversation_store[ID]:
    print(f"{turn['role'].capitalize()}: {turn['content']}")

### Demonstration 2: Follow-up Message (Context Proof)

Send a follow-up question that only makes sense with context from the first message. This proves the injected history actually carries context between requests — the LLM can answer "What is its Population?" because it sees the previous conversation about Paris.

In [ ]:
message = "What is its Population?"
ID = "abs123"
res = test_client.post(
    "/chat",
    params={
        "message": message,
        "session_id": ID
    })

print(f"\nAnswer: {res.json()['answer']}")

print("\nConversation history:")
for turn in conversation_store[ID]:
    print(f"{turn['role'].capitalize()}: {turn['content']}")

### Demonstration 3: Error Handling (yield/finally Guarantee)

Call the endpoint with `simulate_error=True` to raise a `ValueError` **after** the user message is appended to history. The endpoint will crash, but `get_session`'s `finally` block still runs. We wrap the call in `try/except` in this cell (NOT in the endpoint) and print `conversation_store["abs123"]` to prove the user message was recorded despite the failure.

In [ ]:
message = "This causes error, but message is appended to history"
try:
    res = test_client.post(
        "/chat",
        params={"message": message, "session_id": ID, "simulate_error": True}
    )
    print(res.json())
except ValueError as e:
    print(f"Request failed as expected: {e}")

# Proof: the user message was still recorded despite the error
print("\nConversation history:")
for turn in conversation_store[ID]:
    print(f"{turn['role'].capitalize()}: {turn['content']}")

### Demonstration 4: Session Isolation

Start a new session with a different `session_id`. This proves sessions are isolated — the new session has its own empty history, completely separate from `abs123`.

In [ ]:
message = "Hello! What model are you?"
ID = "NewID"
res = test_client.post(
    "/chat",
    params={
        "message": message,
        "session_id": ID
    })

print(f"\nAnswer: {res.json()['answer']}")

print("\nConversation history:")
for turn in conversation_store[ID]:
    print(f"{turn['role'].capitalize()}: {turn['content']}")